In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

In [2]:
builder = SparkSession.builder \
    .appName("Verify-Silver") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
print("=" * 80)
print("SILVER LAYER VERIFICATION")
print("=" * 80)

# Read Silver tables
customers = spark.read.format("delta").load("s3a://silver/customers_silver")
accounts = spark.read.format("delta").load("s3a://silver/accounts_silver")
transactions = spark.read.format("delta").load("s3a://silver/transactions_silver")
aml_alerts = spark.read.format("delta").load("s3a://silver/aml_alerts_silver")

print(f"\n✅ Customers: {customers.count():,} records")
print(f"✅ Accounts: {accounts.count():,} records")
print(f"✅ Transactions: {transactions.count():,} records")
print(f"✅ AML Alerts: {aml_alerts.count():,} records")

# Verify derived columns
print("\n📊 Derived Columns Verification:")

print("\n  Age Band Distribution:")
customers.groupBy("age_band").count().orderBy("count", ascending=False).show()

print("\n  Balance Band Distribution:")
accounts.groupBy("balance_band").count().orderBy("count", ascending=False).show()

print("\n  AML High-Value Flags:")
high_value_txns = transactions.filter("high_value_flag = TRUE")
print(f"  Total high-value transactions: {high_value_txns.count():,}")

print("\n  SLA Breach Alerts:")
sla_breach = aml_alerts.filter("sla_breach_flag = TRUE")
print(f"  Alerts breaching SLA: {sla_breach.count():,}")

SILVER LAYER VERIFICATION

✅ Customers: 10,000 records
✅ Accounts: 28,299 records
✅ Transactions: 95,054 records
✅ AML Alerts: 5,000 records

📊 Derived Columns Verification:

  Age Band Distribution:
+-----------+-----+
|   age_band|count|
+-----------+-----+
|     SENIOR| 3557|
|     MATURE| 2966|
|MIDDLE_AGED| 2363|
|      YOUNG| 1114|
+-----------+-----+


  Balance Band Distribution:
+------------+-----+
|balance_band|count|
+------------+-----+
|      MEDIUM|10006|
|    NEGATIVE| 9104|
|        HIGH| 4733|
|   VERY_HIGH| 2759|
|         LOW| 1697|
+------------+-----+


  AML High-Value Flags:
  Total high-value transactions: 9,538

  SLA Breach Alerts:
  Alerts breaching SLA: 1,899


In [4]:
# Sample join (Customer 360 preview)
print("\n🔍 Sample Customer 360 Query:")
customer_360 = customers.alias("c") \
    .join(accounts.alias("a"), "customer_id") \
    .join(
        transactions.groupBy("account_id").agg(
            {"amount": "sum", "transaction_id": "count"}
        ).alias("t"),
        "account_id",
        "left"
    )

customer_360.select(
    "c.customer_id", "c.first_name", "c.last_name", "c.customer_segment",
    "a.account_id", "a.balance", 
    "t.count(transaction_id)"
).show(5, truncate=False)

print("\n" + "=" * 80)
print("✅ SILVER LAYER VERIFICATION COMPLETE!")
print("=" * 80)

spark.stop()


🔍 Sample Customer 360 Query:
+-----------+----------+---------+----------------+-------------+----------+---------------------+
|customer_id|first_name|last_name|customer_segment|account_id   |balance   |count(transaction_id)|
+-----------+----------+---------+----------------+-------------+----------+---------------------+
|C000001    |JESSICA   |KNIGHT   |RETAIL          |ACCA479800901|-729423.00|null                 |
|C000001    |JESSICA   |KNIGHT   |RETAIL          |ACC6CFAB9D776|5850.00   |4                    |
|C000003    |LAURA     |SCOTT    |RETAIL          |ACC7FEE0EA7CD|-29103.00 |5                    |
|C000003    |LAURA     |SCOTT    |RETAIL          |ACC95DB855B3C|5897.00   |4                    |
|C000004    |SARA      |MURPHY   |RETAIL          |ACCF34405E930|59954.00  |null                 |
+-----------+----------+---------+----------------+-------------+----------+---------------------+
only showing top 5 rows


✅ SILVER LAYER VERIFICATION COMPLETE!
